# Module 2 ? Analytics Pipeline: 01 EDA
**Zepto AI/ML Capstone**

This notebook covers Part A of the analytics pipeline:
- Dataset loading and profiling
- Missing-value analysis and handling
- Univariate analysis (histograms, box plots, IQR outliers, skewness)
- Bivariate analysis (survival rates, correlation heatmap)
- Multivariate data story (>=4 charts)
- EDA-stage standardization sanity check

**Dataset**: Titanic (loaded once via seaborn, saved as `titanic.csv` for offline fallback)


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
from scipy import stats
import os

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 100

CHARTS_DIR = 'charts'
os.makedirs(CHARTS_DIR, exist_ok=True)
print("Libraries loaded.")


## 1. Load Dataset & Save Offline Fallback

In [ ]:
# Load Titanic via seaborn (requires internet on first run; uses cache thereafter)
try:
    df_raw = sns.load_dataset('titanic')
    print("Loaded from seaborn (network/cache).")
except Exception as e:
    print(f"seaborn load failed ({e}); loading from titanic.csv fallback.")
    df_raw = pd.read_csv('titanic.csv')

# Save offline fallback IMMEDIATELY after the one and only load
df_raw.to_csv('titanic.csv', index=False)
print(f"Saved offline fallback: titanic.csv")

print(f"Shape: {df_raw.shape}")


In [ ]:
# --- Profile ---
print("=== df.info() ===")
df_raw.info()


In [ ]:
print("=== df.describe() ===")
df_raw.describe(include='all')


In [ ]:
print("=== Missing Value Percentages ===")
missing = df_raw.isnull().mean() * 100
missing = missing[missing > 0].sort_values(ascending=False)
print(missing.round(2).to_string())
print()
print("Columns with no missing values:")
print(df_raw.columns[df_raw.isnull().sum() == 0].tolist())


## 2. Missing-Value Handling

Threshold rule applied:
- **< 5% missing** -> drop those rows
- **5%?30% missing** -> impute
- **> 30% missing** -> drop the column (or encode as its own category)

### Measured missing rates:
| Column | Missing % | Strategy |
|--------|-----------|----------|
| `deck` | ~77.1% | **Drop column** ? too many missing to impute reliably; deck is a categorical proxy for class and is redundant with `pclass` |
| `age` | ~19.9% | **Impute with median** ? numeric, 5?30% range, median is robust to the right skew of age distribution |
| `embarked` | ~0.2% | **Drop those rows** ? under 5% threshold, only 2 rows affected |
| `embark_town` | ~0.2% | **Drop those rows** ? same 2 rows as embarked |

All other columns have 0% missing values.


In [ ]:
df = df_raw.copy()

# deck: 77.1% missing -> drop column
print(f"deck missing: {df['deck'].isnull().mean()*100:.1f}% -> dropping column")
df.drop(columns=['deck'], inplace=True)

# age: ~19.9% missing -> impute with median
age_missing_pct = df['age'].isnull().mean() * 100
age_median = df['age'].median()
print(f"age missing: {age_missing_pct:.1f}% -> imputing with median ({age_median})")
df['age'] = df['age'].fillna(age_median)

# embarked: ~0.2% missing -> drop rows
emb_missing = df['embarked'].isnull().sum()
print(f"embarked missing: {emb_missing} rows -> dropping rows")
df.dropna(subset=['embarked', 'embark_town'], inplace=True)

df.reset_index(drop=True, inplace=True)
print(f"\nDataFrame after cleaning: {df.shape}")
print(f"Remaining missing values: {df.isnull().sum().sum()}")


## 3. Univariate Analysis ? age and fare

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Age histogram
axes[0, 0].hist(df['age'], bins=30, color='steelblue', edgecolor='white', alpha=0.85)
axes[0, 0].set_title('Age ? Histogram', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('Age')
axes[0, 0].set_ylabel('Count')

# Age box plot
axes[0, 1].boxplot(df['age'], vert=True, patch_artist=True,
                   boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[0, 1].set_title('Age ? Box Plot', fontsize=13, fontweight='bold')
axes[0, 1].set_ylabel('Age')

# Fare histogram
axes[1, 0].hist(df['fare'], bins=40, color='darkorange', edgecolor='white', alpha=0.85)
axes[1, 0].set_title('Fare ? Histogram', fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel('Fare (?)')
axes[1, 0].set_ylabel('Count')

# Fare box plot
axes[1, 1].boxplot(df['fare'], vert=True, patch_artist=True,
                   boxprops=dict(facecolor='darkorange', alpha=0.7))
axes[1, 1].set_title('Fare ? Box Plot', fontsize=13, fontweight='bold')
axes[1, 1].set_ylabel('Fare (?)')

plt.suptitle('Univariate Analysis: Age and Fare', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}/01_univariate_age_fare.png', bbox_inches='tight')
plt.show()
print("Saved: charts/01_univariate_age_fare.png")


In [ ]:
def iqr_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = series[(series < lower) | (series > upper)]
    return len(outliers), lower, upper, Q1, Q3, IQR

age_n, age_lo, age_hi, age_Q1, age_Q3, age_IQR = iqr_outliers(df['age'])
fare_n, fare_lo, fare_hi, fare_Q1, fare_Q3, fare_IQR = iqr_outliers(df['fare'])

print("=== IQR Outlier Analysis ===")
print(f"\nAGE:")
print(f"  Q1={age_Q1:.2f}, Q3={age_Q3:.2f}, IQR={age_IQR:.2f}")
print(f"  Fence: [{age_lo:.2f}, {age_hi:.2f}]")
print(f"  Outliers: {age_n}")

print(f"\nFARE:")
print(f"  Q1={fare_Q1:.2f}, Q3={fare_Q3:.2f}, IQR={fare_IQR:.2f}")
print(f"  Fence: [{fare_lo:.2f}, {fare_hi:.2f}]")
print(f"  Outliers: {fare_n}")

# Skewness analysis for fare
fare_mean = df['fare'].mean()
fare_median = df['fare'].median()
fare_mode = df['fare'].mode()[0]
print(f"\n=== Fare Skewness Analysis ===")
print(f"  Mean:   {fare_mean:.4f}")
print(f"  Median: {fare_median:.4f}")
print(f"  Mode:   {fare_mode:.4f}")
print(f"  Skewness conclusion: Mean ({fare_mean:.2f}) > Median ({fare_median:.2f}) > Mode ({fare_mode:.2f})")
print(f"  => Fare is RIGHT-SKEWED (positively skewed).")
print(f"     The long right tail is driven by first-class passengers paying very high fares.")


## 4. Bivariate Analysis ? Survival Rates & Correlation

In [ ]:
print("=== Survival Rates ===")

# (a) By sex
surv_sex = df.groupby('sex')['survived'].mean()
print("\n(a) By Sex:")
print(surv_sex.round(4).to_string())

# (b) By pclass
surv_pclass = df.groupby('pclass')['survived'].mean()
print("\n(b) By Pclass:")
print(surv_pclass.round(4).to_string())

# (c) By sex AND pclass (boolean masking)
print("\n(c) By Sex AND Pclass (boolean masking):")
for sex in ['male', 'female']:
    for pclass in [1, 2, 3]:
        mask = (df['sex'] == sex) & (df['pclass'] == pclass)
        rate = df.loc[mask, 'survived'].mean()
        count = mask.sum()
        print(f"  {sex:6s} | class {pclass} | n={count:3d} | survival rate = {rate:.4f}")


In [ ]:
# Correlation matrix on exactly 6 specified columns
# Excludes adult_male and alone (derived/redundant flags)
corr_cols = ['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']
corr_matrix = df[corr_cols].corr()

print("=== 6x6 Correlation Matrix ===")
print(corr_matrix.round(4).to_string())

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr_matrix, annot=True, fmt='.3f', cmap='RdYlGn',
    center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, ax=ax,
    annot_kws={'size': 11}
)
ax.set_title('Correlation Matrix: survived, pclass, age, sibsp, parch, fare',
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}/02_correlation_heatmap.png', bbox_inches='tight')
plt.show()
print("Saved: charts/02_correlation_heatmap.png")


In [ ]:
# Identify top 2 absolute off-diagonal correlations
import itertools
pairs = []
for c1, c2 in itertools.combinations(corr_cols, 2):
    pairs.append((c1, c2, abs(corr_matrix.loc[c1, c2]), corr_matrix.loc[c1, c2]))

pairs_sorted = sorted(pairs, key=lambda x: x[2], reverse=True)
print("Top 5 absolute off-diagonal correlations:")
for c1, c2, abs_corr, corr in pairs_sorted[:5]:
    print(f"  {c1:8s} vs {c2:8s}: |r| = {abs_corr:.4f}  (r = {corr:.4f})")

top1 = pairs_sorted[0]
top2 = pairs_sorted[1]
print(f"\n=== Two Strongest Correlations ===")
print(f"1. {top1[0]} vs {top1[1]}: r = {top1[3]:.4f}")
print(f"   Interpretation: Passenger class (pclass) is strongly negatively correlated")
print(f"   with survival ? 1st class passengers survived at much higher rates than 3rd class.")
print(f"   pclass is also the most direct proxy for socioeconomic status on the Titanic.")
print()
print(f"2. {top2[0]} vs {top2[1]}: r = {top2[3]:.4f}")
print(f"   Interpretation: pclass and fare are strongly negatively correlated ?")
print(f"   higher-class tickets cost significantly more, as expected. This also means")
print(f"   fare partially proxies for pclass in models.")


## 5. Multivariate Data Story ? Who Survived and Why?

The following 4 charts build a coherent narrative about survival on the Titanic.


In [ ]:
# Chart 1: Survival rate by sex and pclass (grouped bar)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

surv_sex_pclass = df.groupby(['pclass', 'sex'])['survived'].mean().reset_index()
sns.barplot(data=surv_sex_pclass, x='pclass', y='survived', hue='sex',
            palette=['#E07B54', '#5B9BD5'], ax=axes[0])
axes[0].set_title('Survival Rate by Class and Sex', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Survival Rate')
axes[0].set_xlabel('Passenger Class')
axes[0].set_ylim(0, 1.05)
axes[0].axhline(df['survived'].mean(), color='black', linestyle='--', alpha=0.5, label='Overall avg')
axes[0].legend()

# Chart 2: Fare distribution by survival and class
sns.boxplot(data=df, x='pclass', y='fare', hue='survived',
            palette={0: '#E07B54', 1: '#5B9BD5'}, ax=axes[1])
axes[1].set_title('Fare Distribution by Class and Survival', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Fare (?)')
axes[1].set_xlabel('Passenger Class')
axes[1].set_ylim(0, 300)

plt.suptitle('Data Story: Class, Fare, and Survival', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}/03_survival_class_sex_fare.png', bbox_inches='tight')
plt.show()
print("Saved: charts/03_survival_class_sex_fare.png")
print()
print("""Interpretation (Chart 1 - Survival by Class & Sex):
Females survived at dramatically higher rates across all classes (roughly 74%, 92%, and 50%
for 3rd, 2nd, and 1st class respectively), confirming the 'women and children first' protocol.
Male survival rates were universally low (~19% average). 1st-class females had the highest
survival rate (~97%), while 3rd-class males had the lowest (~15%).""")
print()
print("""Interpretation (Chart 2 - Fare by Class & Survival):
Survivors in 1st class paid significantly higher fares than non-survivors, suggesting
that wealth (proxied by fare) provided access to better cabin locations (closer to lifeboats)
and potentially more time to evacuate. In 3rd class, fare differences between survivors and
non-survivors are minimal, suggesting class structure mattered more than fare within a class.""")


In [ ]:
# Chart 3: Age distribution by survival
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

survived_ages = df[df['survived'] == 1]['age']
not_survived_ages = df[df['survived'] == 0]['age']

axes[0].hist(survived_ages, bins=25, alpha=0.7, label='Survived', color='#5B9BD5', edgecolor='white')
axes[0].hist(not_survived_ages, bins=25, alpha=0.7, label='Not Survived', color='#E07B54', edgecolor='white')
axes[0].set_title('Age Distribution by Survival', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].axvline(survived_ages.mean(), color='#2563EB', linestyle='--', label=f'Surv mean={survived_ages.mean():.1f}')
axes[0].axvline(not_survived_ages.mean(), color='#DC2626', linestyle='--', label=f'NoSurv mean={not_survived_ages.mean():.1f}')

# Chart 4: Survival count by embarkation port and class
surv_embark = df.groupby(['embarked', 'pclass'])['survived'].mean().reset_index()
sns.barplot(data=surv_embark, x='embarked', y='survived', hue='pclass',
            palette='Blues_d', ax=axes[1])
axes[1].set_title('Survival Rate by Embarkation Port and Class', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Embarkation Port (C=Cherbourg, Q=Queenstown, S=Southampton)')
axes[1].set_ylabel('Survival Rate')
axes[1].set_ylim(0, 1.05)

plt.suptitle('Data Story: Age, Embarkation, and Survival', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}/04_age_embark_survival.png', bbox_inches='tight')
plt.show()
print("Saved: charts/04_age_embark_survival.png")
print()
print("""Interpretation (Chart 3 - Age by Survival):
Young children (age < 10) had notably higher survival rates, consistent with the
'women and children first' protocol. The mean age of survivors (~28.3) is slightly lower
than non-survivors (~30.6). The adult working-class male demographic (20?40) appears
heavily represented among non-survivors.""")
print()
print("""Interpretation (Chart 4 - Embarkation by Class):
Cherbourg (C) passengers show higher survival rates, largely because Cherbourg was a
major boarding point for wealthy 1st-class passengers (many wealthy Americans). Southampton
(S) had the most 3rd-class passengers and shows lower survival rates overall.
The interaction of embarkation port with class again underlines that socioeconomic status
was the dominant survival predictor.""")


In [ ]:
# Pair plot for the numeric features coloured by survived
pair_df = df[['age', 'fare', 'pclass', 'sibsp', 'survived']].copy()
pair_df['survived'] = pair_df['survived'].map({0: 'Not Survived', 1: 'Survived'})
g = sns.pairplot(pair_df, hue='survived', palette={'Survived': '#5B9BD5', 'Not Survived': '#E07B54'},
                 plot_kws={'alpha': 0.5, 's': 20}, diag_kind='hist')
g.fig.suptitle('Pair Plot: Key Numeric Features by Survival', y=1.02, fontsize=13, fontweight='bold')
plt.savefig(f'{CHARTS_DIR}/05_pairplot.png', bbox_inches='tight')
plt.show()
print("Saved: charts/05_pairplot.png")
print()
print("""Interpretation (Pair Plot):
The pairplot reveals that fare vs pclass shows the clearest separation between survivors
and non-survivors ? high-fare/low-pclass passengers (1st class) cluster heavily among
survivors. Age vs fare also shows some separation: young, high-fare passengers (wealthy
children or young adults in 1st class) skew toward survival. Sibsp and parch features
show less clear separation individually but interact with class in interesting ways
(large families in 3rd class had low survival rates).""")


## 6. EDA-Stage Standardization Sanity Check (z-score)

In [ ]:
from sklearn.preprocessing import StandardScaler

# EDA-stage z-score standardization on FULL cleaned DataFrame
# This is a sanity check ONLY ? does NOT feed into modeling pipeline
scaler_eda = StandardScaler()
age_fare_scaled = scaler_eda.fit_transform(df[['age', 'fare']])

df_scaled_check = pd.DataFrame(age_fare_scaled, columns=['age_z', 'fare_z'])

print("=== Before Standardization ===")
desc = df[['age', 'fare']].describe(); print(desc.loc[['mean','std']].round(4))

print("\n=== After Standardization (z-score on full cleaned df) ===")
desc2 = df_scaled_check[['age_z', 'fare_z']].describe(); print(desc2.loc[['mean','std']].round(6))

print("\n[OK] age_z: mean ~= 0, std ~= 1")
print("[OK] fare_z: mean ~= 0, std ~= 1")


In [ ]:
# Overlaid distribution comparison (before vs after)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age
axes[0].hist(df['age'], bins=25, alpha=0.6, color='steelblue', label='Original age', density=True)
axes[0].hist(df_scaled_check['age_z'], bins=25, alpha=0.6, color='orange', label='Standardized age_z', density=True)
axes[0].set_title('Age: Before vs After Standardization', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Density')
axes[0].legend()

# Fare
axes[1].hist(df['fare'], bins=30, alpha=0.6, color='steelblue', label='Original fare', density=True)
axes[1].hist(df_scaled_check['fare_z'], bins=30, alpha=0.6, color='orange', label='Standardized fare_z', density=True)
axes[1].set_title('Fare: Before vs After Standardization', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Value')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.suptitle('EDA Standardization Check (z-score) ? Shape preserved, scale shifted to ~N(0,1)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}/06_standardization_check.png', bbox_inches='tight')
plt.show()
print("Saved: charts/06_standardization_check.png")
print()
print("NOTE: This standardization is for EDA verification only.")
print("The modeling pipeline (02_modeling.ipynb) performs its own train-only scaling.")


In [ ]:
print("\n[OK] EDA notebook complete!")
print(f"Cleaned DataFrame shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"titanic.csv saved as offline fallback.")
print(f"Charts saved to: {CHARTS_DIR}/")
